# LUCAS European soil and landscape data

This notebook is in a customized project for inserting all data and metadata available from the European wide soil sampling network [Land Use and Cover Area frame Survey (LUCAS)](https://esdac.jrc.ec.europa.eu/projects/lucas). The project requires customization of the xspatula core framework. By default the path to the start module for a custom project is ./src/"project_name"; it needs to contain a startup function. The default name of that function is "Run_process". 

## Utility data

All required utility data are shipped together with the notebook in the `xspatula_lucas` GitHub repo. This notebook inserts utility data in the database in a **single-step**. Each cell in the `utility` block translates data straight from excel/csv into the database in one step, using the `insert_tabular_data` process. This happens in 2 steps; first the excel data is translated to JSON command files and then the JSON command files are read and compared with the database process records that tell where each data point should be stored in the database. The intermediate JSON files are stored under sub-directories called `staging` (e.g. ./lucas/import_data/utility/general/insert_process/staging). There is no manual hand-off between steps, and this route is **INSERT-only**: existing records are left untouched, never updated. If you need to UPDATE or DELETE data, or want to inspect/edit the generated JSON before it is applied, please consult the [manual](https://xspatula.github.io). In short you can either manually edit the JSON files under `staging` and write a new notebook cell that points to the edited file and rerun it, or you can use a **two-step** process that first translates the tabular data to JSON process files and then use a second manage command after editing the JSON process file.

## LUCAS data

LUCAS specific data are not shipped with the notebook but have to be downloaded after due registration online from [https://esdac.jrc.ec.europa.eu/projects/lucas](https://esdac.jrc.ec.europa.eu/projects/lucas). Once you have accessed the LUCAS data, configure and run the Command Line Interface (CLI) scripts included in the `xspatula_lucas` repo as oulined in the online documentation for both [LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/prepare_data/) and [LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/). The LUCAS data is directly transformed into JSON files that are ready for specific xspatula processes to interpret and insert in the database. If you make changes to the database you need to also edit the CLI's that generate the JSON files to follow any changes made in the table structure.

Note that the overall metadata for LUCAS (`data_source`, `person`, `dataset` and `campaign`) are available as excel files that come with the repo - see [Dataset Metadata Explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/dataset_meta_explained/). This allows you to change the overall naming and definitions of the LUCAS dataset metadata without editing the scripts.

## Data included

This notebook loads all available available LUCAS data - LUCAS 2009 (including Bulgaria, Cyprus, Iceland, Malta and Romania, and the separately recorded total phosphorus data) and LUCAS 2015. Along with the soil observations also landcape (land cover and land use) observations done as part of LUCAS and the biogeographic region of each sample point as recorded in the [LUCAS-Master-Grid.csv](https://ec.europa.eu/eurostat/web/lucas/database/primary-data), freely available online, are included.

## Select data to load

To select which data to include in your own LUCAS framework database, you can run a subset of the notebook cells below. The simplest is to choose the data you do not need by activating/de-activitaing cells from the first code line. The first line in each code cell, `%%script false --no-raise-error`, determines if the cell is being executed or not. If the line is commented out by starting with a hash-tag `#` the cell will execute, remove the hash-tag and it will not.

In [ ]:
# Standard library imports
from os import path, getcwd

import sys

# Get the working directory of the notebook
notebook_path = getcwd()

# Set the path to the root directory of this jupyter notebook (python) project
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:

    sys.path.append(module_path)

# Package application imports
from src.lib import Initiate_process

# Import the custom class for running processes defined in the ai4sh package
from src.ai4sh import Run_process

## Scheme file
You need to set the [scheme_file](https://xspatula.github.io/setup_core_db_docs/framework/scheme_file/) that defines the location (path) to the project, associated process files and your user. You can set the path to the scheme file with a relative path (relative to this notebook), in your home directory using tilde (~) or with an absolute path in the syntax of your operating system. For details see the documentation page on [Xspatula scheme file](https://xspatula.github.io/xspatula_core_docs/framework/scheme_file/).

In [ ]:
# Set the path to the scheme_file

scheme_file = '../scheme_lucas.json'

# 1. Utilitities

All the utlitities (catalogue data) are read from tabular data (excel or CSV) that can be edited before inserted, see the documentation page on [Insert Utility](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/insert_utility/).

## 1.1 Insert general utilities

Inserts general utility data (e.g. territory) in a single step. See the manual page on [Single-step insert](https://xspatula.github.io/xspatula_core_docs/user_data/single_step_insert/) for details on how this, and the following utility related cells, insert excel/CSV data into the database.

**process**: insert_tabular_data

**job_file**: job_insert_general_utility.json

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/utility/job_insert_general_utility.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 1.2 Insert observation utilities

Observation utility data consist of catalogues for quantities, units, methods, data providers and the provisions they offer, and much more. You **must** insert this data before you can enter any actual data, as explained in the documentation page on [Utility Explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/utility_explained/).

This single cell translates the tabular (csv or excel) source data to JSON and immediately applies it to the database, in dependency order (utilities without foreign keys first, then those that depend on them).

**process**: [insert_tabular_data](add_link_to_documentation)

**job_file**: [job_insert_observation_utility.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/utility/job_insert_observation_utility.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 1.3. Insert observation utilities with inheritance

Some observation utilites optionally inherit values from other utility tables. If the option to inheret records is activated these tables must be loaded with data after all the parent tables are filled. For detaies see the manual page on [Utility Explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/utility_explained/).

**process**: [insert_tabular_data](add_link_to_documentation)

**job_file**: [job_insert_observation_utility_inherit.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/utility/job_insert_observation_utility_inherit.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 1.4 Insert landscape utility

Landscape utilities consist of catalogues for defining land use, land cover, land form, biogeography etc. See the manual page on [Insert landscape utility](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/utility_explained/#4-insert-landscape-utility) for details.

**process**: [insert_tabular_data](add_link_to_documentation)

**job_file**: [job_insert_landscape_utility.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/utility/job_insert_landscape_utility.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 2. Dataset metadata

Dataset metdata includes `data_source`, `person`, `dataset` and `campaign` metadata. This data is included as excel files in the `xspatula_lucas` GitHub repo to facilitate editing. The dataset metadate is explained in the manual page on See the documentation page on [Dataset Metadata Explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/dataset_meta_explained).

## 2.1 Insert data source

Inserts the data source (organisation or individual controlling the dataset). See the documentation on [Dataset Metadata Explained - Data source](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/dataset_meta_explained/#data-source).

**process**: [insert_tabular_data](add_link_to_documentation)

**process_file**: [data_source.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/dataset_meta/insert_process/data_source.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## 2.2 Insert persons

Inserts the person(s) responsible and involved in data collection and analysis. See the documentation on [Dataset Metadata Explained - Person](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/dataset_meta_explained/#person).

**process**: [insert_tabular_data](add_link_to_documentation)

**process_file**: [person.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/dataset_meta/insert_process/person.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## 2.3 Insert dataset

Inserts the dataset (umbrella for data observations from a source, usually with a distinct thematic content). See the documentation on [Dataset Metadata Explained - Dataset](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/dataset_meta_explained/#dataset).

**process**: [insert_tabular_data](add_link_to_documentation)

**process_file**: [dataset.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/dataset_meta/insert_process/dataset.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

## 2.4 Insert campaigns

Inserts the campaign(s) (limited sampling and observation operation for data collection belonging to a predefined dataset). See the documentation on [Dataset Metadata Explained - Campaign](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/dataset_meta_explained/#campaign).

**process**: [insert_tabular_data](add_link_to_documentation)

**process_file**: [campaign.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
process_file = 'import_data/dataset_meta/insert_process/campaign.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:
    Run_process(structured_process_D, scheme_params_D)

# 3. LUCAS 2009 data

The complete processing pipeline is outlined in the documentation collection [LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/). Before you get started you must access the actual LUCAS 2009 data that is freely available under a registration page as described in the documentation page [Prepare data](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/prepare_data/). Once downloaded you must follow the instructions for [Configure and run the script](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/prepare_data/#2-configure-and-run-the-script) included in the `LUCAS 2009` project repo for converting the LUCAS 2009 data to a format that the framework can handle. The sequential processes to call are summarised in the documentation page [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/).

## 3.1 Load LUCAS 2009 sampling log

Inserts the sampling log for LUCAS 2009. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/).

**process**: [manage_sampling_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_sampling_log.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_sampling_log.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.2 Load LUCAS 2009 geolocations

Inserts the geographic location for LUCAS 2009 samples. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Samples Explained - Geolocation](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/samples_explained/#geolocation).

**process**: [manage_geolocation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_geolocation.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_geolocation.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.3 Load LUCAS 2009 samples

Inserts the LUCAS 2009 samples. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Samples Explained - Sample](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/samples_explained/#sample).

**process**: [manage_geolocated_profile_sample](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_sample.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_sample.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.4 LUCAS 2009 laboratory data

The LUCAS 2009 soil data include both wet laboratory observations and spectral observation. To load the wet laboratory data you must first load the observation log and then the observations, see the documentation page on [Observations explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/) and its section on [laboratory observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#laboratory-observations).

### 3.4.1 Load LUCAS 2009 laboratory observation log

Inserts the LUCAS 2009 laboratory observation log. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Observation log](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#laboratory-observation-log).


**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_log_lab](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_log_lab.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.4.2 Load LUCAS 2009 laboratory observations

Inserts the LUCAS 2009 laboratory observations. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Observation](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#laboratory-observations).


**process**: [manage_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_lab.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_lab.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.5 LUCAS 2009 spectral data

The LUCAS 2009 soil data include both wet laboratory observations and spectral observation. To load the spectral data you must first load the details on the spectrometer used followed by the observation log and then the observations, see the documentation page on [Observations explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/) and its section on [spectral observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectral-observations).

### 3.5.1 Load LUCAS 2009 spectrometer

Inserts the LUCAS 2009 spectrometer. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Spectrometer](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectrometer).


**process**: [manage_spectrometer](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_spectrometer.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_spectrometer.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.5.2 Load LUCAS 2009 spectra observation log

Inserts the LUCAS 2009 spectra observation log. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Spectra observation log](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectra-observation-log).


**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_log_spectra.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_log_spectra.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.5.3 Load LUCAS 2009 spectra observations

Inserts the LUCAS 2009 spectral data. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Spectra observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectra-observations).


**process**: [manage_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_spectra.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_spectra.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.6 LUCAS 2009 landscape data

The LUCAS 2009 data also include different landscape observation - derived either by manual classification in the field or from maps. To load the landscape data you must first load the observation logs and then observations as outlined on [Observations explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/) and its section on [landscape observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#landscape-observations).

### 3.6.1 Load LUCAS 2009 landscape obervation logs

Inserts the LUCAS 2009 shared observation log for land use and land cover. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Landscape observation log](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#landsacape-observation-log).


**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_log_landscape.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_log_landscape.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.6.2 Load LUCAS 2009 land cover

Inserts the LUCAS 2009 land cover observations. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Land cover and land use](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#land-cover-and-land-use).


**process**: [manage_land_cover_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_land_cover.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_land_cover.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.6.3 Load LUCAS 2009 land use

Inserts the LUCAS 2009 land use observations. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Land cover and land use](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#land-cover-and-land-use).


**process**: [manage_land_use_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_land_use.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_land_use.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 3.7 LUCAS 2009 biogeographic regions (BIOGE16)

The [BIOGEO16 biogeogphical data](https://www.eea.europa.eu/en/datahub/datahubitem-view/11db8d14-f167-4cd5-9205-95638dfd9618) stem not from LUCAS, but from the [European Environment Agancy](https://www.eea.europa.eu/en).

### 3.7.1 Load LUCAS 2009 BIOGEO16 sampling log

BIOGEO16 should already be defined as both a dataset and a campaign, but also the sampling log must be defined. The sampling log was produced by the script that translated your downloaded LUCAS 2019 data.

Insert the BIOGEO16 sampling log from the next cell. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [NOT YET AVAILABLE](#).


**process**: [manage_sampling_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_sampling_log_biogeo.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_sampling_log_biogeo.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.7.2 Load BIOGEO16 obervation log

Inserts the BIOGEO16 observation log. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [NOT YET AVAILABLE](#).

**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_log_biogeo.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_log_biogeo.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 3.7.3 Load LUCAS 2009 Biogeographic regions

Inserts the BIOGEO16 bioegeographic region data for LUCAS 2009 sample locations. See the documentation page on [Load LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/load_lucas_2009/) with details on the page [Observations Explained - Biogeographic region (BIOGEO16) observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#biogeographic-region-biogeo16-observations).


**process**: [manage_biogeo16_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_biogeo.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2009/job_LUCAS_2009_observation_biogeo.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# 4 LUCAS 2015 data

The complete processing pipeline is outlined in the documentation collection [LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/). Before you get started you must access the actual LUCAS 2015 data that is freely available under a registration page as described in the documentation page on [LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/). Once downloaded you must follow the instructions for [Configure and run the script](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/prepare_data/#2-configure-and-run-the-script) included in the `LUCAS 2015` project for converting the LUCAS 2015 data to a format that the framework can handle. The sequential processes to call are summarised in the documentation page [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/).

The data loading explanations and details are the same as for LUCAS 2009 and not repeated for LUCAS 2015. The [LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/) only summarises the steps and refers back to [LUCAS 2009](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/) for details.

## 4.1 Load LUCAS 2015 sampling log

Inserts the sampling log for LUCAS 2015. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015).

**process**: [manage_sampling_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_sampling_log.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_sampling_log.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 4.2 Load LUCAS 2015 geolocations

Inserts the geographic location for LUCAS 2015 samples. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Samples Explained - Geolocation](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/samples_explained/#geolocation).

**process**: [manage_geolocation](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_geolocation.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_geolocation.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 4.3 Load LUCAS 2015 samples

Inserts the samples for LUCAS 2015 samples. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Samples Explained - Sample](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/samples_explained/#sample).

**process**: [manage_geolocation](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_samples.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_sample.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 4.4 LUCAS 2015 laboratory data

The LUCAS 2015 soil data include both wet laboratory observations and spectral observation. To load the wet laboratory data you must first load the observation log and then the observations, see the LUCAS 2009 documentation page on [Observations explained](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/) and its section on [laboratory observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#laboratory-observations).

### 4.4.1 Load LUCAS 2015 laboratory observation log

Inserts the samples for LUCAS 2015 samples. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Observation log](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#laboratory-observation-log).

**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_observation_log_lab](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_observation_log_lab.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.4.2. Load LUCAS 2015 laboratory observations

Inserts the LUCAS 2015 laboratory observations. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the page [Observations Explained - Observation](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#laboratory-observations).


**process**: [manage_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2009_observation_lab.json](add_link_to_documentation)

## 4.5 LUCAS 2015 spectral data

The LUCAS 2015 soil data include both wet laboratory observations and spectral observation. To load the spectral data you must first load the details on the spectrometer used followed by the observation log and then the observations, see the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015).

### 4.5.1 Load LUCAS 2015 spectrometer

Inserts the LUCAS 2015 spectrometer. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Spectrometer](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectrometer).

**process**: [manage_spectrometer](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_spectrometer.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_spectrometer.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.5.2 Load LUCAS 2015 spectra observation log

Inserts the LUCAS 2015 spectra observation log. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Spectra observation log](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectra-observation-log).

**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_observation_log_spectra.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_observation_log_spectra.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.5.2 Load LUCAS 2015 spectra observations

Inserts the LUCAS 2015 spectral data. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Spectra observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#spectra-observations).


**process**: [manage_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_observation_spectra.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_observation_spectra.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 4.6 LUCAS 2015 landscape data

The LUCAS 2015 data also include different landscape observation - derived either by manual classification in the field or from maps. To load the landscape data you must first load the observation logs and then observations as outlined in [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details in the LUCAS 2009 pages.

### 4.6.1 Load LUCAS 2015 landscape obervation logs

Inserts the LUCAS 2015 shared observation log for land use and land cover. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Landscape observation log](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#landsacape-observation-log).


**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_observation_log_landscape.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_observation_log_landscape.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.6.2 Load LUCAS 2015 land cover

Inserts the LUCAS 2015 land cover observations. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Land cover and land use](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#land-cover-and-land-use).


**process**: [manage_land_cover_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_land_cover.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_land_cover.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.6.3 Load LUCAS 2015 land use

Inserts the LUCAS 2015 land use observations. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Land cover and land use](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#land-cover-and-land-use).


**process**: [manage_land_cover_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_land_use.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_land_use.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

## 4.7 LUCAS 2015 biogeographic regions (BIOGE16)

The [BIOGEO16 biogeogphical data](https://www.eea.europa.eu/en/datahub/datahubitem-view/11db8d14-f167-4cd5-9205-95638dfd9618) stem not from LUCAS, but from the [European Environment Agancy](https://www.eea.europa.eu/en).

### 4.7.1 Load LUCAS 2009 BIOGEO16 sampling log

BIOGEO16 should already be defined as both a dataset and a campaign, but also the sampling_log must be defined. The sampling log was produced by the script that translated your downloaded LUCAS 2019 data.

Insert the BIOGEO16 sampling log from the next cell. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the page [NOT YET AVAILABLE](#).


**process**: [manage_sampling_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_sampling_log_biogeo.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_sampling_log_biogeo.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.7.2 Load BIOGEO16 obervation log

Inserts the BIOGEO16 observation log. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the page [NOT YET AVAILABLE](#).


**process**: [manage_observation_log](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_observation_log_biogeo.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_observation_log_biogeo.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

### 4.7.3 Load LUCAS 2015 Biogeographic regions

Inserts the BIOGEO16 bioegeographic region data for LUCAS 2015 sample locations. See the documentation page on [Load LUCAS 2015](https://xspatula.github.io/xspatula_lucas_docs/lucas_2015/#4-load-lucas-2015) with details on the LUCAS 2009 page [Observations Explained - Biogeographic region (BIOGEO16) observations](https://xspatula.github.io/xspatula_lucas_docs/lucas_2009/observations_explained/#biogeographic-region-biogeo16-observations).


**process**: [manage_biogeo16_observation](add_link_to_documentation)

**job_file**: [job_LUCAS_2015_observation_biogeo.json](add_link_to_documentation)

In [ ]:
#%%script false --no-raise-error
job_file = 'import_data/LUCAS_2015/job_LUCAS_2015_observation_biogeo.json'
structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, job_file)

if structured_process_D is not None:
    
    Run_process(structured_process_D, scheme_params_D)

# NOTE - the first records of LUCAS 2015 represent UK samples and BIOGEO16 does not cover the UK, 
# if you get ❌ ERROR the path to the json process file(s) does not exist: ...
# this is probably because you imported the LUCAS 2015 with too few records. 